In [2]:
!git clone https://github.com/SundaramPandey2005/LOTA-AI-Image-Forensics.git

Cloning into 'LOTA-AI-Image-Forensics'...
remote: Enumerating objects: 105, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 105 (delta 9), reused 105 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (105/105), 3.92 MiB | 9.18 MiB/s, done.
Resolving deltas: 100% (9/9), done.


In [3]:
%cd /content/LOTA-AI-Image-Forensics

/content/LOTA-AI-Image-Forensics


In [4]:
!ls

Additional_prompt.md  docs		 prompt_2.1.md	   scripts
app		      experiments	 prompt_2.2.md	   src
app.py		      notebooks		 README.md	   tests
configs		      Project_prompt.md  requirements.txt  Things_to_improve.md


In [5]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 105.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 120.3 MB/s eta 0:00:00


In [6]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [7]:
!python scripts/run_infrastructure_check.py

  LOTA INFRASTRUCTURE GATE (GATE A)
  Mode: Mock / Software Infrastructure Validation (use_mock_data=True)

  [CHECK 1] Environment & Device : PASSED (cuda, PyTorch 2.11.0+cu128)
  [CHECK 2] Dataset Construction : PASSED (Mock samples: 32 train, 16 val)
  [CHECK 3] Model Initialization : PASSED (ResNet-50 NBC)
  [CHECK 4] Forward & Backward   : PASSED (Loss: 0.8134)
  [CHECK 5] Checkpoint Save/Load : PASSED
[EXPERIMENT LOGGED] Successfully recorded run 'EXP_INFRASTRUCTURE_GATE_CHECK' [source: mock_fixture, mock: True] to SQLite database (./experiments/results/lota_experiments.db).
  [CHECK 6] Database Logging     : PASSED (Recorded with is_mock=1)
---------------------------------------------------------------------------
  INFRASTRUCTURE GATE: PASSED
---------------------------------------------------------------------------
  [NOTE] Software infrastructure is fully operational.
         This gate validates code execution only and does NOT imply real-data readiness.



In [8]:
!python scripts/run_pilot_check.py --gate both

  LOTA INFRASTRUCTURE GATE (GATE A)
  Mode: Mock / Software Infrastructure Validation (use_mock_data=True)

  [CHECK 1] Environment & Device : PASSED (cuda, PyTorch 2.11.0+cu128)
  [CHECK 2] Dataset Construction : PASSED (Mock samples: 32 train, 16 val)
  [CHECK 3] Model Initialization : PASSED (ResNet-50 NBC)
  [CHECK 4] Forward & Backward   : PASSED (Loss: 0.7436)
  [CHECK 5] Checkpoint Save/Load : PASSED
[EXPERIMENT LOGGED] Successfully recorded run 'EXP_INFRASTRUCTURE_GATE_CHECK' [source: mock_fixture, mock: True] to SQLite database (./experiments/results/lota_experiments.db).
  [CHECK 6] Database Logging     : PASSED (Recorded with is_mock=1)
---------------------------------------------------------------------------
  INFRASTRUCTURE GATE: PASSED
---------------------------------------------------------------------------
  [NOTE] Software infrastructure is fully operational.
         This gate validates code execution only and does NOT imply real-data readiness.

  LOTA REAL-DATA 

In [9]:
# In Google Colab:
!git pull
!python scripts/run_infrastructure_check.py
!python scripts/run_pilot_check.py --gate both

Already up to date.
  LOTA INFRASTRUCTURE GATE (GATE A)
  Mode: Mock / Software Infrastructure Validation (use_mock_data=True)

  [CHECK 1] Environment & Device : PASSED (cuda, PyTorch 2.11.0+cu128)
  [CHECK 2] Dataset Construction : PASSED (Mock samples: 32 train, 16 val)
  [CHECK 3] Model Initialization : PASSED (ResNet-50 NBC)
  [CHECK 4] Forward & Backward   : PASSED (Loss: 0.8285)
  [CHECK 5] Checkpoint Save/Load : PASSED
[EXPERIMENT LOGGED] Successfully recorded run 'EXP_INFRASTRUCTURE_GATE_CHECK' [source: mock_fixture, mock: True] to SQLite database (./experiments/results/lota_experiments.db).
  [CHECK 6] Database Logging     : PASSED (Recorded with is_mock=1)
---------------------------------------------------------------------------
  INFRASTRUCTURE GATE: PASSED
---------------------------------------------------------------------------
  [NOTE] Software infrastructure is fully operational.
         This gate validates code execution only and does NOT imply real-data readiness

In [1]:
# Phase 2 — Real GenImage Data Pilot
!pip install -q kaggle

In [10]:
import os

os.environ["KAGGLE_USERNAME"] = "Sundaram_Pandey2024"
os.environ["KAGGLE_KEY"] = "KGAT_97d6d92320ad9fac10e0590246bc5d20"

In [11]:
!kaggle datasets list -s tiny-genimage

ref                                  title                            size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-----------------------------------  -------------------------  ----------  --------------------------  -------------  ---------  ---------------  
yangsangtai/tiny-genimage            tiny genimage              8345888545  2024-01-22 15:34:57.707000           2771         18  0.6875           
cartografia/unbiased-tiny-genimage   Unbiased Tiny GenImage     2525870358  2026-01-21 14:17:18.617000           6938          2  0.9411765        
renhuang8/genimage-subset-detection  Genimage_Subset_Detection  3623734761  2025-11-30 21:06:32.100000            639          4  0.5625           
priyanshipatel11/genimage21          genimage21                 8346868561  2026-04-30 16:28:36.810000              0          1  0.0625           


In [12]:
!mkdir -p /content/genimage_download

!kaggle datasets download \
    -d yangsangtai/tiny-genimage \
    -p /content/genimage_download

Dataset URL: https://www.kaggle.com/datasets/yangsangtai/tiny-genimage
License(s): CC-BY-NC-SA-4.0
100% 7.77G/7.77G [01:32<00:00, 89.8MB/s]



In [13]:
!ls -lh /content/genimage_download

total 7.8G
-rw-r--r-- 1 root root 7.8G Jan 22  2024 tiny-genimage.zip


In [14]:
!unzip -l /content/genimage_download/tiny-genimage.zip | head -100

Archive:  /content/genimage_download/tiny-genimage.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
    32865  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/000_biggan_00093.png
    29827  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/001_biggan_00033.png
    19276  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/002_biggan_00010.png
    18024  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/002_biggan_00163.png
    14701  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/003_biggan_00052.png
    18849  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/003_biggan_00058.png
    22685  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/004_biggan_00054.png
    20412  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/004_biggan_00068.png
    16970  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/005_biggan_00086.png
    20717  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/006_biggan_00005.png
    33692  2024-01-22 15:38   image

In [15]:
!unzip -l /content/genimage_download/tiny-genimage.zip | grep -E "nature|ai" | head -50

    32865  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/000_biggan_00093.png
    29827  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/001_biggan_00033.png
    19276  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/002_biggan_00010.png
    18024  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/002_biggan_00163.png
    14701  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/003_biggan_00052.png
    18849  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/003_biggan_00058.png
    22685  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/004_biggan_00054.png
    20412  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/004_biggan_00068.png
    16970  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/005_biggan_00086.png
    20717  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/006_biggan_00005.png
    33692  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/006_biggan_00097.png
    11308  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/006_biggan_00

In [16]:
!unzip -l /content/genimage_download/tiny-genimage.zip | \
awk '{print $4}' | \
grep '^imagenet_ai_0419_biggan/val/' | \
head -30

imagenet_ai_0419_biggan/val/ai/002_biggan_00127.png
imagenet_ai_0419_biggan/val/ai/002_biggan_00143.png
imagenet_ai_0419_biggan/val/ai/003_biggan_00035.png
imagenet_ai_0419_biggan/val/ai/006_biggan_00035.png
imagenet_ai_0419_biggan/val/ai/006_biggan_00039.png
imagenet_ai_0419_biggan/val/ai/008_biggan_00039.png
imagenet_ai_0419_biggan/val/ai/008_biggan_00127.png
imagenet_ai_0419_biggan/val/ai/011_biggan_00035.png
imagenet_ai_0419_biggan/val/ai/011_biggan_00074.png
imagenet_ai_0419_biggan/val/ai/014_biggan_00035.png
imagenet_ai_0419_biggan/val/ai/014_biggan_00143.png
imagenet_ai_0419_biggan/val/ai/016_biggan_00020.png
imagenet_ai_0419_biggan/val/ai/016_biggan_00127.png
imagenet_ai_0419_biggan/val/ai/016_biggan_00143.png
imagenet_ai_0419_biggan/val/ai/018_biggan_00020.png
imagenet_ai_0419_biggan/val/ai/020_biggan_00143.png
imagenet_ai_0419_biggan/val/ai/023_biggan_00143.png
imagenet_ai_0419_biggan/val/ai/024_biggan_00127.png
imagenet_ai_0419_biggan/val/ai/031_biggan_00143.png
imagenet_ai_

In [17]:
!unzip -l /content/genimage_download/tiny-genimage.zip | \
awk '{print $4}' | \
grep '^imagenet_ai_0419_biggan/val/nature/' | \
head -10

imagenet_ai_0419_biggan/val/nature/ILSVRC2012_val_00000035.JPEG
imagenet_ai_0419_biggan/val/nature/ILSVRC2012_val_00000062.JPEG
imagenet_ai_0419_biggan/val/nature/ILSVRC2012_val_00000198.JPEG
imagenet_ai_0419_biggan/val/nature/ILSVRC2012_val_00000351.JPEG
imagenet_ai_0419_biggan/val/nature/ILSVRC2012_val_00000372.JPEG
imagenet_ai_0419_biggan/val/nature/ILSVRC2012_val_00000386.JPEG
imagenet_ai_0419_biggan/val/nature/ILSVRC2012_val_00000424.JPEG
imagenet_ai_0419_biggan/val/nature/ILSVRC2012_val_00000670.JPEG
imagenet_ai_0419_biggan/val/nature/ILSVRC2012_val_00000696.JPEG
imagenet_ai_0419_biggan/val/nature/ILSVRC2012_val_00000814.JPEG


In [18]:
!unzip -l /content/genimage_download/tiny-genimage.zip | \
awk '{print $4}' | \
grep '^imagenet_ai_0419_biggan/val/ai/' | \
head -10

imagenet_ai_0419_biggan/val/ai/002_biggan_00127.png
imagenet_ai_0419_biggan/val/ai/002_biggan_00143.png
imagenet_ai_0419_biggan/val/ai/003_biggan_00035.png
imagenet_ai_0419_biggan/val/ai/006_biggan_00035.png
imagenet_ai_0419_biggan/val/ai/006_biggan_00039.png
imagenet_ai_0419_biggan/val/ai/008_biggan_00039.png
imagenet_ai_0419_biggan/val/ai/008_biggan_00127.png
imagenet_ai_0419_biggan/val/ai/011_biggan_00035.png
imagenet_ai_0419_biggan/val/ai/011_biggan_00074.png
imagenet_ai_0419_biggan/val/ai/014_biggan_00035.png


In [19]:
%cd /content/LOTA-AI-Image-Forensics

/content/LOTA-AI-Image-Forensics


In [20]:
!mkdir -p data/GenImage/biggan/val/nature
!mkdir -p data/GenImage/biggan/val/ai

In [21]:
import zipfile
import os

zip_path = "/content/genimage_download/tiny-genimage.zip"
output_dir = "/content/LOTA-AI-Image-Forensics/data/GenImage/biggan/val/nature"

with zipfile.ZipFile(zip_path, "r") as z:
    real_files = [
        name for name in z.namelist()
        if name.startswith("imagenet_ai_0419_biggan/val/nature/")
        and not name.endswith("/")
    ][:100]

    for file in real_files:
        filename = os.path.basename(file)
        with z.open(file) as source, open(os.path.join(output_dir, filename), "wb") as target:
            target.write(source.read())

print(f"Extracted {len(real_files)} real images")

Extracted 100 real images


In [22]:
import zipfile
import os

zip_path = "/content/genimage_download/tiny-genimage.zip"
output_dir = "/content/LOTA-AI-Image-Forensics/data/GenImage/biggan/val/ai"

with zipfile.ZipFile(zip_path, "r") as z:
    fake_files = [
        name for name in z.namelist()
        if name.startswith("imagenet_ai_0419_biggan/val/ai/")
        and not name.endswith("/")
    ][:100]

    for file in fake_files:
        filename = os.path.basename(file)
        with z.open(file) as source, open(os.path.join(output_dir, filename), "wb") as target:
            target.write(source.read())

print(f"Extracted {len(fake_files)} AI-generated images")

Extracted 100 AI-generated images


In [23]:
!echo "Real images:"
!find data/GenImage/biggan/val/nature -type f | wc -l

!echo "AI images:"
!find data/GenImage/biggan/val/ai -type f | wc -l

!echo ""
!find data/GenImage/biggan -type f | head

Real images:
100
AI images:
100

data/GenImage/biggan/val/ai/071_biggan_00074.png
data/GenImage/biggan/val/ai/130_biggan_00127.png
data/GenImage/biggan/val/ai/006_biggan_00035.png
data/GenImage/biggan/val/ai/051_biggan_00020.png
data/GenImage/biggan/val/ai/149_biggan_00127.png
data/GenImage/biggan/val/ai/126_biggan_00127.png
data/GenImage/biggan/val/ai/002_biggan_00127.png
data/GenImage/biggan/val/ai/006_biggan_00039.png
data/GenImage/biggan/val/ai/122_biggan_00039.png
data/GenImage/biggan/val/ai/166_biggan_00074.png


In [24]:
!python scripts/check_data_readiness.py

  LOTA REAL-DATA VALIDATION READINESS GATE
Target Dataset Root: /content/LOTA-AI-Image-Forensics/data/GenImage

  [CHECK 1] Dataset Root Directory : FOUND (./data/GenImage)
  [CHECK 2] Generator Folders      : FOUND (['biggan'])
Traceback (most recent call last):
  File "/content/LOTA-AI-Image-Forensics/scripts/check_data_readiness.py", line 80, in <module>
    check_genimage_readiness()
    ~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/content/LOTA-AI-Image-Forensics/scripts/check_data_readiness.py", line 41, in check_genimage_readiness
    ds = GenImageDataset(root_dir=root_dir, split="val", use_mock_data=False)
  File "/content/LOTA-AI-Image-Forensics/src/data/dataset.py", line 80, in __init__
    self._discover_samples(max_samples_per_class)
    ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/LOTA-AI-Image-Forensics/src/data/dataset.py", line 112, in _discover_samples
    raise RuntimeError(
    ...<4 lines>...
    )
RuntimeError: No valid image samples discovered under '/cont

In [25]:
!grep -R "sd15" -n scripts src configs app.py 2>/dev/null

scripts/check_data_readiness.py:31:        print("     data/GenImage/sd15/val/nature/   (Real images, e.g. 0_nature.png)")
scripts/check_data_readiness.py:32:        print("     data/GenImage/sd15/val/ai/       (AI images, e.g. 0_ai.png)")
scripts/seed_reference_data.py:22:        ("Table 1", "LOTA-nbc", "sd15", "biggan", 100.0, 1.000, 1.000, "Paper Table 1, Row: LOTA-nbc"),
scripts/seed_reference_data.py:23:        ("Table 1", "LOTA-nbc", "sd15", "sd14", 99.9, 0.999, 0.999, "Paper Table 1, Row: LOTA-nbc"),
scripts/seed_reference_data.py:24:        ("Table 1", "LOTA-nbc", "sd15", "sd15", 99.9, 0.999, 0.999, "Paper Table 1, Row: LOTA-nbc"),
scripts/seed_reference_data.py:25:        ("Table 1", "LOTA-nbc", "sd15", "midjourney", 93.1, 0.962, 0.958, "Paper Table 1, Row: LOTA-nbc"),
scripts/seed_reference_data.py:26:        ("Table 1", "LOTA-nbc", "sd15", "adm", 99.7, 0.999, 0.999, "Paper Table 1, Row: LOTA-nbc (Corrected from 98.5 -> 99.7)"),
scripts/seed_reference_data.py:27:        ("Tab

In [26]:
!sed -n '1,180p' scripts/check_data_readiness.py

"""
Real-Data Readiness Gate for LOTA
Verifies whether an actual GenImage dataset subset is available for real-world forensic validation and training.
"""
import os
import sys
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))

from PIL import Image
from src.data.dataset import GenImageDataset


def check_genimage_readiness(root_dir: str = "./data/GenImage") -> bool:
    print("=" * 75)
    print("  LOTA REAL-DATA VALIDATION READINESS GATE")
    print("=" * 75)
    print(f"Target Dataset Root: {os.path.abspath(root_dir)}\n")

    if not os.path.exists(root_dir):
        print("  [CHECK 1] Dataset Root Directory : MISSING")
        print("  [CHECK 2] Generator Folders      : MISSING")
        print("  [CHECK 3] Real Nature Images     : MISSING")
        print("  [CHECK 4] Fake AI Images         : MISSING")
        print("  [CHECK 5] Image Loading Test     : FAILED\n")
        print("-" * 75)
        print("  REAL-DATA VALIDATION STATUS: NOT READY")
      

In [27]:
!sed -n '1,220p' src/data/dataset.py

import os
import glob
from typing import List, Dict, Optional, Tuple, Union, Any
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np

from src.data.preprocessing import get_transforms, preprocess_raw_image, extract_lota_forensic_features
from src.forensic.bitplanes import compose_low_bit_planes
from src.forensic.normalization import normalize_noise_thresholding, normalize_noise_scaling
from src.forensic.mgps import maximum_gradient_patch_selection


class GenImageDataset(Dataset):
    """
    PyTorch Dataset for GenImage with native forensic low-bit plane pipeline support.
    
    Supports:
    - Multi-generator training (e.g. for LOGO training sets).
    - Single-generator evaluation.
    - Deterministic subsampling (for pilot checks and compute-constrained testing).
    - Explicit mock dataset mode for unit testing and CI only (use_mock_data=True).
    
    STRICT RESEARCH INTEGRITY POLICY:
    - If use_mock_data=False (the default)

In [28]:
from pathlib import Path

path = Path("scripts/check_data_readiness.py")

text = path.read_text()

old = '''    ds = GenImageDataset(root_dir=root_dir, split="val", use_mock_data=False)
    if ds.use_mock_data or len(ds.samples) < 2:
        print("  [CHECK 3] Sample Population      : INSUFFICIENT (< 2 real/fake files)")
        print("-" * 75)
        print("  REAL-DATA VALIDATION STATUS: NOT READY")
        print("-" * 75)
        return False

    # Check real and fake existence
    has_real = any(s["label"] == 0 for s in ds.samples)
    has_fake = any(s["label"] == 1 for s in ds.samples)
'''

new = '''    # Explicitly use the generator folders actually discovered above.
    # This prevents the dataset class from falling back to its default "sd15".
    ds = GenImageDataset(
        root_dir=root_dir,
        generators=generators,
        split="val",
        use_mock_data=False
    )

    if ds.use_mock_data or len(ds.samples) < 2:
        print("  [CHECK 3] Sample Population      : INSUFFICIENT (< 2 real/fake files)")
        print("-" * 75)
        print("  REAL-DATA VALIDATION STATUS: NOT READY")
        print("-" * 75)
        return False

    # ds.samples contains tuples:
    # (file_path, label, generator)
    has_real = any(label == 0 for _, label, _ in ds.samples)
    has_fake = any(label == 1 for _, label, _ in ds.samples)
'''

if old not in text:
    raise RuntimeError("Expected code block was not found. No changes were made.")

path.write_text(text.replace(old, new))

print("check_data_readiness.py updated successfully.")

check_data_readiness.py updated successfully.


In [29]:
!sed -n '95,145p' scripts/check_data_readiness.py

In [30]:
!python scripts/check_data_readiness.py

  LOTA REAL-DATA VALIDATION READINESS GATE
Target Dataset Root: /content/LOTA-AI-Image-Forensics/data/GenImage

  [CHECK 1] Dataset Root Directory : FOUND (./data/GenImage)
  [CHECK 2] Generator Folders      : FOUND (['biggan'])
  [CHECK 3] Real Nature Images     : MISSING
  [CHECK 4] Fake AI Images         : FOUND
  [CHECK 5] Image Loading Test     : PASSED (Shape: torch.Size([3, 256, 256]))
---------------------------------------------------------------------------
  REAL-DATA VALIDATION STATUS: NOT READY
---------------------------------------------------------------------------


In [31]:
from src.data.dataset import GenImageDataset

ds = GenImageDataset(
    root_dir="./data/GenImage",
    generators=["biggan"],
    split="val",
    use_mock_data=False
)

print("Total samples:", len(ds.samples))
print()

for sample in ds.samples[:10]:
    print(sample)

print("\nLabel counts:")
from collections import Counter
print(Counter(label for _, label, _ in ds.samples))

Total samples: 100

('./data/GenImage/biggan/val/ai/002_biggan_00127.png', 1.0, 'biggan')
('./data/GenImage/biggan/val/ai/002_biggan_00143.png', 1.0, 'biggan')
('./data/GenImage/biggan/val/ai/003_biggan_00035.png', 1.0, 'biggan')
('./data/GenImage/biggan/val/ai/006_biggan_00035.png', 1.0, 'biggan')
('./data/GenImage/biggan/val/ai/006_biggan_00039.png', 1.0, 'biggan')
('./data/GenImage/biggan/val/ai/008_biggan_00039.png', 1.0, 'biggan')
('./data/GenImage/biggan/val/ai/008_biggan_00127.png', 1.0, 'biggan')
('./data/GenImage/biggan/val/ai/011_biggan_00035.png', 1.0, 'biggan')
('./data/GenImage/biggan/val/ai/011_biggan_00074.png', 1.0, 'biggan')
('./data/GenImage/biggan/val/ai/014_biggan_00035.png', 1.0, 'biggan')

Label counts:
Counter({1.0: 100})


In [32]:
for path, label, generator in ds.samples:
    if "/nature/" in path:
        print("Nature example:", path, "Label:", label)
        break

for path, label, generator in ds.samples:
    if "/ai/" in path:
        print("AI example:", path, "Label:", label)
        break

AI example: ./data/GenImage/biggan/val/ai/002_biggan_00127.png Label: 1.0


In [33]:
!find data/GenImage/biggan/val/nature -type f | head -10

data/GenImage/biggan/val/nature/ILSVRC2012_val_00007767.JPEG
data/GenImage/biggan/val/nature/ILSVRC2012_val_00008496.JPEG
data/GenImage/biggan/val/nature/ILSVRC2012_val_00006772.JPEG
data/GenImage/biggan/val/nature/ILSVRC2012_val_00009058.JPEG
data/GenImage/biggan/val/nature/ILSVRC2012_val_00005324.JPEG
data/GenImage/biggan/val/nature/ILSVRC2012_val_00001240.JPEG
data/GenImage/biggan/val/nature/ILSVRC2012_val_00007795.JPEG
data/GenImage/biggan/val/nature/ILSVRC2012_val_00000670.JPEG
data/GenImage/biggan/val/nature/ILSVRC2012_val_00008635.JPEG
data/GenImage/biggan/val/nature/ILSVRC2012_val_00003634.JPEG


In [34]:
!grep -n -A 40 -B 10 "discover_samples" src/data/dataset.py

70-                label = float(i % 2)
71-                self.samples.append((f"mock_{gen}_{split}_{i}.png", label, gen))
72-        else:
73-            # Real dataset mode: must exist and have samples
74-            if not os.path.exists(root_dir):
75-                raise FileNotFoundError(
76-                    f"GenImage dataset directory was not found at '{os.path.abspath(root_dir)}'.\n"
77-                    f"Real-data training cannot proceed without legitimate image files.\n"
78-                    f"If you intentionally wish to test software infrastructure with synthetic data, set use_mock_data=True."
79-                )
80:            self._discover_samples(max_samples_per_class)
81-
82:    def _discover_samples(self, max_samples_per_class: Optional[int]):
83-        real_exts = ("*.jpg", "*.jpeg", "*.png", "*.webp")
84-        discovered_by_gen = {}
85-
86-        for gen in self.generators:
87-            discovered_by_gen[gen] = {"nature": 0, "ai": 0}
88-            

In [36]:
from pathlib import Path

path = Path("src/data/dataset.py")
text = path.read_text()

old = '''real_exts = ("*.jpg", "*.jpeg", "*.png", "*.webp")'''

new = '''real_exts = ("*.jpg", "*.jpeg", "*.JPEG", "*.png", "*.PNG", "*.webp", "*.WEBP")'''

if old not in text:
    raise RuntimeError("Expected extension definition was not found.")

path.write_text(text.replace(old, new))

print("dataset.py updated successfully.")

dataset.py updated successfully.


In [37]:
!grep -n "real_exts" src/data/dataset.py

83:        real_exts = ("*.jpg", "*.jpeg", "*.JPEG", "*.png", "*.PNG", "*.webp", "*.WEBP")
99:                        for ext in real_exts:


In [38]:
from src.data.dataset import GenImageDataset
from collections import Counter

ds = GenImageDataset(
    root_dir="./data/GenImage",
    generators=["biggan"],
    split="val",
    use_mock_data=False
)

print("Total samples:", len(ds.samples))
print("Label counts:", Counter(label for _, label, _ in ds.samples))

for path, label, generator in ds.samples:
    if "/nature/" in path:
        print("\nNature example:", path, "Label:", label)
        break

for path, label, generator in ds.samples:
    if "/ai/" in path:
        print("AI example:", path, "Label:", label)
        break

Total samples: 100
Label counts: Counter({1.0: 100})
AI example: ./data/GenImage/biggan/val/ai/002_biggan_00127.png Label: 1.0


In [39]:
import glob

nature_dir = "./data/GenImage/biggan/val/nature"

print("Lowercase .jpeg:", len(glob.glob(f"{nature_dir}/*.jpeg")))
print("Uppercase .JPEG:", len(glob.glob(f"{nature_dir}/*.JPEG")))
print("All files:", len(glob.glob(f"{nature_dir}/*")))

print("\nFirst few uppercase matches:")
print(glob.glob(f"{nature_dir}/*.JPEG")[:5])

Lowercase .jpeg: 0
Uppercase .JPEG: 100
All files: 100

First few uppercase matches:
['./data/GenImage/biggan/val/nature/ILSVRC2012_val_00007767.JPEG', './data/GenImage/biggan/val/nature/ILSVRC2012_val_00008496.JPEG', './data/GenImage/biggan/val/nature/ILSVRC2012_val_00006772.JPEG', './data/GenImage/biggan/val/nature/ILSVRC2012_val_00009058.JPEG', './data/GenImage/biggan/val/nature/ILSVRC2012_val_00005324.JPEG']


In [40]:
import os

print("GenImage:", os.listdir("./data/GenImage"))
print("BigGAN:", os.listdir("./data/GenImage/biggan"))
print("VAL:", os.listdir("./data/GenImage/biggan/val"))

GenImage: ['biggan']
BigGAN: ['val']
VAL: ['ai', 'nature']


In [41]:
import importlib
import src.data.dataset as dataset_module

importlib.reload(dataset_module)

GenImageDataset = dataset_module.GenImageDataset

from collections import Counter

ds = GenImageDataset(
    root_dir="./data/GenImage",
    generators=["biggan"],
    split="val",
    use_mock_data=False
)

print("Total samples:", len(ds.samples))
print("Label counts:", Counter(label for _, label, _ in ds.samples))

for path, label, generator in ds.samples:
    if "/nature/" in path:
        print("\nNature example:", path, "Label:", label)
        break

for path, label, generator in ds.samples:
    if "/ai/" in path:
        print("AI example:", path, "Label:", label)
        break

Total samples: 200
Label counts: Counter({0.0: 100, 1.0: 100})

Nature example: ./data/GenImage/biggan/val/nature/ILSVRC2012_val_00000035.JPEG Label: 0.0
AI example: ./data/GenImage/biggan/val/ai/002_biggan_00127.png Label: 1.0


In [42]:
!python scripts/check_data_readiness.py

  LOTA REAL-DATA VALIDATION READINESS GATE
Target Dataset Root: /content/LOTA-AI-Image-Forensics/data/GenImage

  [CHECK 1] Dataset Root Directory : FOUND (./data/GenImage)
  [CHECK 2] Generator Folders      : FOUND (['biggan'])
  [CHECK 3] Real Nature Images     : FOUND
  [CHECK 4] Fake AI Images         : FOUND
  [CHECK 5] Image Loading Test     : PASSED (Shape: torch.Size([3, 256, 256]))
---------------------------------------------------------------------------
  REAL-DATA VALIDATION STATUS: READY
---------------------------------------------------------------------------
[READY] 200 real/fake samples successfully indexed across 1 generators.


In [43]:
%cd /content/LOTA-AI-Image-Forensics

/content/LOTA-AI-Image-Forensics


In [44]:
!git pull origin main

remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 13 (delta 12), reused 13 (delta 12), pack-reused 0 (from 0)
Unpacking objects: 100% (13/13), 2.28 KiB | 388.00 KiB/s, done.
From https://github.com/SundaramPandey2005/LOTA-AI-Image-Forensics
 * branch            main       -> FETCH_HEAD
   7dcce04..519b729  main       -> origin/main
Updating 7dcce04..519b729
error: Your local changes to the following files would be overwritten by merge:
	scripts/check_data_readiness.py
	src/data/dataset.py
Please commit your changes or stash them before you merge.
Aborting


In [45]:
!git log --oneline -3

7dcce04 (HEAD -> main) Fix .gitignore and track src/data package
835fb29 Initial LOTA implementation


In [46]:
%cd /content/LOTA-AI-Image-Forensics

!git status
!git branch -a
!git remote -v
!git fetch origin
!git log origin/main --oneline -5

/content/LOTA-AI-Image-Forensics
On branch main
Your branch is behind 'origin/main' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   scripts/check_data_readiness.py
	modified:   src/data/dataset.py

no changes added to commit (use "git add" and/or "git commit -a")
* main
  remotes/origin/HEAD -> origin/main
  remotes/origin/main
origin	https://github.com/SundaramPandey2005/LOTA-AI-Image-Forensics.git (fetch)
origin	https://github.com/SundaramPandey2005/LOTA-AI-Image-Forensics.git (push)
519b729 (origin/main, origin/HEAD) Prepare BigGAN real-data pilot
7dcce04 (HEAD -> main) Fix .gitignore and track src/data package
835fb29 Initial LOTA implementation


In [47]:
!git diff

diff --git a/scripts/check_data_readiness.py b/scripts/check_data_readiness.py
index 885ad83..4cdb960 100644
--- a/scripts/check_data_readiness.py
+++ b/scripts/check_data_readiness.py
@@ -38,7 +38,15 @@ def check_genimage_readiness(root_dir: str = "./data/GenImage") -> bool:
     print(f"  [CHECK 1] Dataset Root Directory : FOUND ({root_dir})")
     print(f"  [CHECK 2] Generator Folders      : FOUND ({generators})")
 
-    ds = GenImageDataset(root_dir=root_dir, split="val", use_mock_data=False)
+    # Explicitly use the generator folders actually discovered above.
+    # This prevents the dataset class from falling back to its default "sd15".
+    ds = GenImageDataset(
+        root_dir=root_dir,
+        generators=generators,
+        split="val",
+        use_mock_data=False
+    )
+
     if ds.use_mock_data or len(ds.samples) < 2:
         print("  [CHECK 3] Sample Population      : INSUFFICIENT (< 2 real/fake files)")
         print("-" * 75)
@@ -46,9 +54,10 @@ def check_genimag

In [48]:
!git stash

Saved working directory and index state WIP on main: 7dcce04 Fix .gitignore and track src/data package


In [49]:
!git pull origin main

From https://github.com/SundaramPandey2005/LOTA-AI-Image-Forensics
 * branch            main       -> FETCH_HEAD
Updating 7dcce04..519b729
Fast-forward
 configs/pilot_real_data.yaml    |  5 +--
 scripts/check_data_readiness.py | 13 ++++++--
 scripts/run_real_data_pilot.py  | 72 +++++++++++++++++++++++++++++------------
 src/data/dataset.py             | 39 +++++++++++++---------
 src/data/splits.py              |  7 ++--
 tests/test_data.py              | 10 ++++++
 6 files changed, 103 insertions(+), 43 deletions(-)


In [50]:
!git log --oneline -5
!git status

519b729 (HEAD -> main, origin/main, origin/HEAD) Prepare BigGAN real-data pilot
7dcce04 Fix .gitignore and track src/data package
835fb29 Initial LOTA implementation
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [51]:
!git stash list

stash@{0}: WIP on main: 7dcce04 Fix .gitignore and track src/data package


In [53]:
!git stash drop "stash@{0}"

error: stash@0 is not a valid reference


In [54]:
!git stash list
!git status

stash@{0}: WIP on main: 7dcce04 Fix .gitignore and track src/data package
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [55]:
!find data/GenImage/biggan/val/nature -type f | wc -l
!find data/GenImage/biggan/val/ai -type f | wc -l

100
100


In [56]:
!python scripts/check_data_readiness.py

  LOTA REAL-DATA VALIDATION READINESS GATE
Target Dataset Root: /content/LOTA-AI-Image-Forensics/data/GenImage

  [CHECK 1] Dataset Root Directory : FOUND (./data/GenImage)
  [CHECK 2] Generator Folders      : FOUND (['biggan'])
  [CHECK 3] Real Nature Images     : FOUND
  [CHECK 4] Fake AI Images         : FOUND
  [CHECK 5] Image Loading Test     : PASSED (Shape: torch.Size([3, 256, 256]))
---------------------------------------------------------------------------
  REAL-DATA VALIDATION STATUS: READY
---------------------------------------------------------------------------
[READY] 200 real/fake samples successfully indexed across 1 generators.


In [57]:
!python scripts/run_real_data_pilot.py --config configs/pilot_real_data.yaml

  LOTA REAL-DATA PILOT GATE (GATE B)
  LOTA REAL-DATA VALIDATION READINESS GATE
Target Dataset Root: /content/LOTA-AI-Image-Forensics/data/GenImage

  [CHECK 1] Dataset Root Directory : FOUND (./data/GenImage)
  [CHECK 2] Generator Folders      : FOUND (['biggan'])
  [CHECK 3] Real Nature Images     : FOUND
  [CHECK 4] Fake AI Images         : FOUND
  [CHECK 5] Image Loading Test     : PASSED (Shape: torch.Size([3, 256, 256]))
---------------------------------------------------------------------------
  REAL-DATA VALIDATION STATUS: READY
---------------------------------------------------------------------------
[READY] 200 real/fake samples successfully indexed across 1 generators.

[DATASET LOADING] Initializing real GenImage dataset...
  Generator Target     : biggan
  Real Train Samples   : 70
  Fake Train Samples   : 70
  Real Val Samples     : 30
  Fake Val Samples     : 30
  Image Size           : 256x256 (MGPS 32x32 Patch)
  Batch Size           : 16
  Compute Device       : cu

In [58]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [60]:
import sqlite3

conn = sqlite3.connect("experiments/results/lota_experiments.db")

cur = conn.cursor()

rows = cur.execute(
    "SELECT * FROM experiments ORDER BY rowid DESC LIMIT 5"
).fetchall()

for row in rows:
    print(row)

conn.close()

('real_data_pilot_biggan', 'Real-Data Pilot BIGGAN', '2026-08-27 12:03:15', 'experimental', 0, 'main', 'M_NBC_RESNET50', 'nbc', None, '{"experiment_name": "real_data_pilot_biggan", "generator": "biggan", "data": {"root_dir": "./data/GenImage", "use_mock_data": false, "train_val_ratio": 0.7, "max_real_samples": 100, "max_fake_samples": 100, "image_size": 256, "patch_size": 32, "bit_planes": [0, 1, 2], "normalization": "thresholding"}, "model": {"architecture": "nbc", "backbone": "resnet50", "pretrained": true, "num_classes": 1}, "training": {"batch_size": 16, "epochs": 3, "learning_rate": 0.0001, "weight_decay": 0.0001, "mixed_precision": true, "checkpoint_dir": "./checkpoints"}, "reproducibility": {"seed": 42, "deterministic": true}}', 'COMPLETED', 0.0)
('EXP_INFRASTRUCTURE_GATE_CHECK', 'Infrastructure Gate Verification', '2026-08-27 11:17:56', 'mock_fixture', 1, 'main', 'M_NBC_RESNET50', 'nbc', None, '{"mode": "infra_check", "mock": true}', 'COMPLETED', 0.0)


In [61]:
%cd /content/LOTA-AI-Image-Forensics

!git pull origin main

/content/LOTA-AI-Image-Forensics
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 15 (delta 9), reused 14 (delta 8), pack-reused 0 (from 0)
Unpacking objects: 100% (15/15), 25.99 KiB | 283.00 KiB/s, done.
From https://github.com/SundaramPandey2005/LOTA-AI-Image-Forensics
 * branch            main       -> FETCH_HEAD
   519b729..2586d6f  main       -> origin/main
Updating 519b729..2586d6f
Fast-forward
 configs/biggan_constrained_baseline_e1.yaml |   34 +
 notebooks/LOTA_AI_Image_Forensics.ipynb     | 2622 +++++++++++++++++++++++++++
 scripts/run_real_data_pilot.py              |  267 ++-
 src/experiments/logger.py                   |    3 +-
 src/training/metrics.py                     |   45 +-
 5 files changed, 2904 insertions(+), 67 deletions(-)
 create mode 100644 configs/biggan_constrained_baseline_e1.yaml
 create mode 100644 notebooks/LOTA_AI_Image_Forensics.ipynb


In [62]:
!git log --oneline -3

2586d6f (HEAD -> main, origin/main, origin/HEAD) Prepare E1 BigGAN constrained baseline
05bfb91 Add Colab real-data experiment notebook
519b729 Prepare BigGAN real-data pilot


In [63]:
%cd /content/LOTA-AI-Image-Forensics

!python scripts/run_real_data_pilot.py --config configs/biggan_constrained_baseline_e1.yaml

/content/LOTA-AI-Image-Forensics
  LOTA REAL-DATA EXPERIMENT RUNNER
  LOTA REAL-DATA VALIDATION READINESS GATE
Target Dataset Root: /content/LOTA-AI-Image-Forensics/data/GenImage

  [CHECK 1] Dataset Root Directory : FOUND (./data/GenImage)
  [CHECK 2] Generator Folders      : FOUND (['biggan'])
  [CHECK 3] Real Nature Images     : FOUND
  [CHECK 4] Fake AI Images         : FOUND
  [CHECK 5] Image Loading Test     : PASSED (Shape: torch.Size([3, 256, 256]))
---------------------------------------------------------------------------
  REAL-DATA VALIDATION STATUS: READY
---------------------------------------------------------------------------
[READY] 200 real/fake samples successfully indexed across 1 generators.

[DATASET LOADING] Initializing real GenImage dataset...

  [RESEARCH INTEGRITY ERROR] INSUFFICIENT SAMPLES FOR CONSTRAINED BASELINE
  Experiment 'biggan_constrained_baseline_e1' requires strict minimum dataset counts (require_exact_sample_counts=True).
  - Genuine (nature) im

In [64]:
import os
from pathlib import Path

root = Path("./data/GenImage/biggan")

print("Directory structure:\n")

for path in sorted(root.rglob("*")):
    if path.is_dir():
        try:
            count = len([
                f for f in path.iterdir()
                if f.is_file()
            ])
            if count > 0:
                print(f"{path}  -->  {count} direct files")
        except:
            pass


Directory structure:

data/GenImage/biggan/val/ai  -->  100 direct files
data/GenImage/biggan/val/nature  -->  100 direct files


In [65]:
from pathlib import Path
from collections import Counter

root = Path("./data/GenImage/biggan")

image_exts = {".jpg", ".jpeg", ".png", ".webp"}

all_images = [
    f for f in root.rglob("*")
    if f.is_file() and f.suffix.lower() in image_exts
]

print("TOTAL IMAGE FILES:", len(all_images))

# Show counts by parent directory
counts = Counter(str(f.parent) for f in all_images)

print("\nIMAGE COUNTS BY DIRECTORY:\n")

for directory, count in sorted(counts.items()):
    print(f"{directory}: {count}")

TOTAL IMAGE FILES: 200

IMAGE COUNTS BY DIRECTORY:

data/GenImage/biggan/val/ai: 100
data/GenImage/biggan/val/nature: 100


In [66]:
!find data/GenImage -type f | head -20

data/GenImage/biggan/val/ai/071_biggan_00074.png
data/GenImage/biggan/val/ai/130_biggan_00127.png
data/GenImage/biggan/val/ai/006_biggan_00035.png
data/GenImage/biggan/val/ai/051_biggan_00020.png
data/GenImage/biggan/val/ai/149_biggan_00127.png
data/GenImage/biggan/val/ai/126_biggan_00127.png
data/GenImage/biggan/val/ai/002_biggan_00127.png
data/GenImage/biggan/val/ai/006_biggan_00039.png
data/GenImage/biggan/val/ai/122_biggan_00039.png
data/GenImage/biggan/val/ai/166_biggan_00074.png
data/GenImage/biggan/val/ai/003_biggan_00035.png
data/GenImage/biggan/val/ai/132_biggan_00074.png
data/GenImage/biggan/val/ai/051_biggan_00127.png
data/GenImage/biggan/val/ai/109_biggan_00035.png
data/GenImage/biggan/val/ai/008_biggan_00039.png
data/GenImage/biggan/val/ai/130_biggan_00143.png
data/GenImage/biggan/val/ai/056_biggan_00020.png
data/GenImage/biggan/val/ai/020_biggan_00143.png
data/GenImage/biggan/val/ai/103_biggan_00020.png
data/GenImage/biggan/val/ai/114_biggan_00143.png


In [67]:
!find data/GenImage -type f \( -iname "*.jpg" -o -iname "*.jpeg" -o -iname "*.png" -o -iname "*.webp" \) | wc -l

200


In [68]:
import os
from collections import defaultdict

root = "data/GenImage"
counts = defaultdict(int)

for dirpath, _, filenames in os.walk(root):
    image_count = sum(
        f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
        for f in filenames
    )
    if image_count:
        counts[dirpath] += image_count

for path, count in sorted(counts.items()):
    print(f"{count:6d}  {path}")

   100  data/GenImage/biggan/val/ai
   100  data/GenImage/biggan/val/nature


In [69]:
!find /content -type f \( -name "*.zip" -o -name "*.tar" -o -name "*.tar.gz" -o -name "*.tgz" \) -printf "%p\n" 2>/dev/null

/content/genimage_download/tiny-genimage.zip


In [70]:
!ls -lh /content/genimage_download

total 7.8G
-rw-r--r-- 1 root root 7.8G Jan 22  2024 tiny-genimage.zip


In [71]:
!unzip -l /content/genimage_download/tiny-genimage.zip | head -30

Archive:  /content/genimage_download/tiny-genimage.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
    32865  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/000_biggan_00093.png
    29827  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/001_biggan_00033.png
    19276  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/002_biggan_00010.png
    18024  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/002_biggan_00163.png
    14701  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/003_biggan_00052.png
    18849  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/003_biggan_00058.png
    22685  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/004_biggan_00054.png
    20412  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/004_biggan_00068.png
    16970  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/005_biggan_00086.png
    20717  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/006_biggan_00005.png
    33692  2024-01-22 15:38   image

In [72]:
!unzip -l /content/genimage_download/tiny-genimage.zip | grep -i "biggan" | head -30

    32865  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/000_biggan_00093.png
    29827  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/001_biggan_00033.png
    19276  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/002_biggan_00010.png
    18024  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/002_biggan_00163.png
    14701  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/003_biggan_00052.png
    18849  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/003_biggan_00058.png
    22685  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/004_biggan_00054.png
    20412  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/004_biggan_00068.png
    16970  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/005_biggan_00086.png
    20717  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/006_biggan_00005.png
    33692  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/006_biggan_00097.png
    11308  2024-01-22 15:38   imagenet_ai_0419_biggan/train/ai/006_biggan_00

In [73]:
!unzip -l /content/genimage_download/tiny-genimage.zip | grep -i "biggan" | wc -l

5000


In [76]:
import zipfile
import os
import shutil

zip_path = "/content/genimage_download/tiny-genimage.zip"
extract_root = "/content/extracted_biggan"

os.makedirs(extract_root, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    ai_files = sorted([
        name for name in z.namelist()
        if name.startswith("imagenet_ai_0419_biggan/train/ai/")
        and name.lower().endswith((".png", ".jpg", ".jpeg"))
    ])[:500]

    print("AI files selected:", len(ai_files))

    for name in ai_files:
        z.extract(name, extract_root)

print("AI extraction complete.")

AI files selected: 500
AI extraction complete.


In [77]:
import zipfile
import os

zip_path = "/content/genimage_download/tiny-genimage.zip"
extract_root = "/content/extracted_biggan"

with zipfile.ZipFile(zip_path, "r") as z:
    nature_files = sorted([
        name for name in z.namelist()
        if name.startswith("imagenet_ai_0419_biggan/train/nature/")
        and name.lower().endswith((".png", ".jpg", ".jpeg"))
    ])[:500]

    print("Nature files selected:", len(nature_files))

    for name in nature_files:
        z.extract(name, extract_root)

print("Nature extraction complete.")

Nature files selected: 500
Nature extraction complete.


In [78]:
import os
import shutil

source = "/content/extracted_biggan/imagenet_ai_0419_biggan/train"
destination = "/content/LOTA-AI-Image-Forensics/data/GenImage/biggan/train"

os.makedirs(destination, exist_ok=True)

for class_name in ["ai", "nature"]:
    src = os.path.join(source, class_name)
    dst = os.path.join(destination, class_name)

    os.makedirs(dst, exist_ok=True)

    for filename in os.listdir(src):
        shutil.copy2(
            os.path.join(src, filename),
            os.path.join(dst, filename)
        )

    print(f"{class_name}: {len(os.listdir(dst))} images")

ai: 500 images
nature: 500 images


In [79]:
!find data/GenImage/biggan/train/ai -type f | wc -l
!find data/GenImage/biggan/train/nature -type f | wc -l

500
500


In [80]:
import os

for split in ["train", "val"]:
    for cls in ["ai", "nature"]:
        path = f"data/GenImage/biggan/{split}/{cls}"
        count = len([
            f for f in os.listdir(path)
            if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))
        ]) if os.path.exists(path) else 0

        print(f"{split:5} | {cls:6} | {count} images")

train | ai     | 500 images
train | nature | 500 images
val   | ai     | 100 images
val   | nature | 100 images


In [81]:
!python scripts/check_data_readiness.py

  LOTA REAL-DATA VALIDATION READINESS GATE
Target Dataset Root: /content/LOTA-AI-Image-Forensics/data/GenImage

  [CHECK 1] Dataset Root Directory : FOUND (./data/GenImage)
  [CHECK 2] Generator Folders      : FOUND (['biggan'])
  [CHECK 3] Real Nature Images     : FOUND
  [CHECK 4] Fake AI Images         : FOUND
  [CHECK 5] Image Loading Test     : PASSED (Shape: torch.Size([3, 256, 256]))
---------------------------------------------------------------------------
  REAL-DATA VALIDATION STATUS: READY
---------------------------------------------------------------------------
[READY] 200 real/fake samples successfully indexed across 1 generators.


In [82]:
from src.data.dataset import GenImageDataset
from collections import Counter

train_ds = GenImageDataset(
    root_dir="./data/GenImage",
    generators=["biggan"],
    split="train",
    use_mock_data=False
)

val_ds = GenImageDataset(
    root_dir="./data/GenImage",
    generators=["biggan"],
    split="val",
    use_mock_data=False
)

print("TRAIN")
print("Total:", len(train_ds))
print("Labels:", Counter(label for _, label, _ in train_ds.samples))

print("\nVALIDATION")
print("Total:", len(val_ds))
print("Labels:", Counter(label for _, label, _ in val_ds.samples))

TRAIN
Total: 1000
Labels: Counter({0.0: 500, 1.0: 500})

VALIDATION
Total: 200
Labels: Counter({0.0: 100, 1.0: 100})


In [83]:
!python scripts/run_real_data_pilot.py \
    --config configs/biggan_constrained_baseline_e1.yaml

  LOTA REAL-DATA EXPERIMENT RUNNER
  LOTA REAL-DATA VALIDATION READINESS GATE
Target Dataset Root: /content/LOTA-AI-Image-Forensics/data/GenImage

  [CHECK 1] Dataset Root Directory : FOUND (./data/GenImage)
  [CHECK 2] Generator Folders      : FOUND (['biggan'])
  [CHECK 3] Real Nature Images     : FOUND
  [CHECK 4] Fake AI Images         : FOUND
  [CHECK 5] Image Loading Test     : PASSED (Shape: torch.Size([3, 256, 256]))
---------------------------------------------------------------------------
  REAL-DATA VALIDATION STATUS: READY
---------------------------------------------------------------------------
[READY] 200 real/fake samples successfully indexed across 1 generators.

[DATASET LOADING] Initializing real GenImage dataset...
  Experiment ID        : biggan_constrained_baseline_e1
  Generator Target     : biggan
  Real Train Samples   : 500
  Fake Train Samples   : 500
  Real Val Samples     : 100
  Fake Val Samples     : 100
  Image Size           : 256x256 (MGPS 32x32 Patc

In [84]:
import os

db_path = "experiments/results/lota_experiments.db"

print("Database exists:", os.path.exists(db_path))

if os.path.exists(db_path):
    print("Size:", os.path.getsize(db_path), "bytes")

Database exists: True
Size: 49152 bytes


In [85]:
import sqlite3

db_path = "experiments/results/lota_experiments.db"

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Show all experiments
cursor.execute("""
SELECT experiment_id, name, created_at, source_type, is_mock, status
FROM experiments
""")

rows = cursor.fetchall()

for row in rows:
    print(row)

conn.close()

OperationalError: no such column: created_at

In [86]:
import sqlite3

db_path = "./experiments/results/lota_experiments.db"

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Show the structure of the experiments table
cursor.execute("PRAGMA table_info(experiments);")

columns = cursor.fetchall()

for column in columns:
    print(column)

conn.close()

(0, 'experiment_id', 'TEXT', 0, None, 1)
(1, 'name', 'TEXT', 1, None, 0)
(2, 'timestamp', 'DATETIME', 0, 'CURRENT_TIMESTAMP', 0)
(3, 'source_type', 'TEXT', 0, "'experimental'", 0)
(4, 'is_mock', 'BOOLEAN', 0, '0', 0)
(5, 'git_commit', 'TEXT', 0, "'main'", 0)
(6, 'model_id', 'TEXT', 0, None, 0)
(7, 'architecture', 'TEXT', 0, None, 0)
(8, 'excluded_generator', 'TEXT', 0, None, 0)
(9, 'config_json', 'TEXT', 0, None, 0)
(10, 'status', 'TEXT', 0, "'COMPLETED'", 0)
(11, 'training_time_sec', 'REAL', 0, '0.0', 0)


In [87]:
import sqlite3

db_path = "./experiments/results/lota_experiments.db"

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute("""
SELECT
    experiment_id,
    name,
    timestamp,
    source_type,
    is_mock,
    git_commit,
    model_id,
    architecture,
    status,
    training_time_sec
FROM experiments
ORDER BY timestamp DESC;
""")

rows = cursor.fetchall()

for row in rows:
    print(row)

conn.close()

('biggan_constrained_baseline_e1', 'E1 BigGAN Constrained Baseline', '2026-08-27 12:51:06', 'experimental', 0, 'main', 'M_NBC_RESNET50', 'nbc', 'COMPLETED', 356.4473030567169)
('real_data_pilot_biggan', 'Real-Data Pilot BIGGAN', '2026-08-27 12:03:15', 'experimental', 0, 'main', 'M_NBC_RESNET50', 'nbc', 'COMPLETED', 0.0)
('EXP_INFRASTRUCTURE_GATE_CHECK', 'Infrastructure Gate Verification', '2026-08-27 11:17:56', 'mock_fixture', 1, 'main', 'M_NBC_RESNET50', 'nbc', 'COMPLETED', 0.0)


In [88]:
import sqlite3

db_path = "./experiments/results/lota_experiments.db"

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# First inspect the metrics table structure
cursor.execute("PRAGMA table_info(metrics);")

for column in cursor.fetchall():
    print(column)

conn.close()

(0, 'id', 'INTEGER', 0, None, 1)
(1, 'experiment_id', 'TEXT', 1, None, 0)
(2, 'generator_id', 'TEXT', 1, None, 0)
(3, 'split', 'TEXT', 0, "'test'", 0)
(4, 'source_type', 'TEXT', 0, "'experimental'", 0)
(5, 'is_mock', 'BOOLEAN', 0, '0', 0)
(6, 'is_unseen', 'BOOLEAN', 0, '0', 0)
(7, 'accuracy', 'REAL', 0, None, 0)
(8, 'auroc', 'REAL', 0, None, 0)
(9, 'average_precision', 'REAL', 0, None, 0)
(10, 'f1', 'REAL', 0, None, 0)
(11, 'precision', 'REAL', 0, None, 0)
(12, 'recall', 'REAL', 0, None, 0)


In [89]:
import sqlite3
import pandas as pd

db_path = "./experiments/results/lota_experiments.db"

conn = sqlite3.connect(db_path)

query = """
SELECT
    experiment_id,
    generator_id,
    split,
    source_type,
    is_mock,
    is_unseen,
    accuracy,
    auroc,
    average_precision,
    f1,
    precision,
    recall
FROM metrics
WHERE experiment_id = 'biggan_constrained_baseline_e1';
"""

df = pd.read_sql_query(query, conn)

conn.close()

print(df.to_string(index=False))

                 experiment_id generator_id     split  source_type  is_mock  is_unseen  accuracy   auroc  average_precision       f1  precision  recall
biggan_constrained_baseline_e1       biggan  val_best experimental        0          0     0.870 0.94565           0.922818 0.879630   0.818966    0.95
biggan_constrained_baseline_e1       biggan val_final experimental        0          0     0.855 0.94110           0.931144 0.851282   0.873684    0.83


In [90]:
import torch
import os

checkpoint_path = "./checkpoints/biggan_constrained_baseline_e1_best.pth"

print("Checkpoint exists:", os.path.exists(checkpoint_path))

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("\nBest epoch:", checkpoint["best_epoch"])

print("\nValidation metrics:")
for key, value in checkpoint["validation_metrics"].items():
    print(f"{key}: {value}")

Checkpoint exists: True

Best epoch: 14

Validation metrics:
accuracy: 0.87
ap: 0.9228181616547592
average_precision: 0.9228181616547592
auroc: 0.94565
f1: 0.8796296296296297
precision: 0.8189655172413793
recall: 0.95
train_loss: 0.04993418056704104
val_loss: 0.42666200421750544


In [91]:
import os

paths = [
    "/content/LOTA-AI-Image-Forensics/checkpoints",
    "/content/LOTA-AI-Image-Forensics/experiments/results",
]

for path in paths:
    print(f"\n📁 {path}")

    if os.path.exists(path):
        for file in os.listdir(path):
            full_path = os.path.join(path, file)
            print(f"  {file} | {os.path.getsize(full_path):,} bytes")
    else:
        print("  ❌ Folder not found")


📁 /content/LOTA-AI-Image-Forensics/checkpoints
  biggan_constrained_baseline_e1_best.pth | 282,599,413 bytes

📁 /content/LOTA-AI-Image-Forensics/experiments/results
  lota_experiments.db | 49,152 bytes


In [92]:
import os

E1_DIR = "/content/E1_RESULTS"

os.makedirs(E1_DIR, exist_ok=True)

print("E1 results folder created:")
print(E1_DIR)


E1 results folder created:
/content/E1_RESULTS


In [93]:
import shutil
import os

repo = "/content/LOTA-AI-Image-Forensics"
e1_dir = "/content/E1_RESULTS"

files_to_copy = {
    "checkpoint": (
        f"{repo}/checkpoints/biggan_constrained_baseline_e1_best.pth",
        f"{e1_dir}/biggan_constrained_baseline_e1_best.pth"
    ),

    "database": (
        f"{repo}/experiments/results/lota_experiments.db",
        f"{e1_dir}/lota_experiments.db"
    ),

    "config": (
        f"{repo}/configs/biggan_constrained_baseline_e1.yaml",
        f"{e1_dir}/biggan_constrained_baseline_e1.yaml"
    )
}

for name, (src, dst) in files_to_copy.items():
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"✅ Copied {name}")
    else:
        print(f"❌ Missing {name}: {src}")

✅ Copied checkpoint
✅ Copied database
✅ Copied config


In [94]:
summary = """
============================================================
LOTA AI IMAGE FORENSICS
EXPERIMENT E1 — BIGGAN CONSTRAINED BASELINE
============================================================

Experiment ID:
biggan_constrained_baseline_e1

Dataset:
Generator: BigGAN

Training Samples:
500 AI-generated images
500 Real/Nature images

Training Configuration:
Epochs: 15
Batch Size: 16
Optimizer: Adam
Learning Rate: 0.0001
Weight Decay: 0.0001
Backbone: ResNet50
Architecture: NBC / LOTA Noise Classifier
Image Size: 256x256
Seed: 42

------------------------------------------------------------
BEST MODEL RESULTS
------------------------------------------------------------

Best Epoch: 14

Validation Accuracy:
0.8700

Validation AUROC:
0.94565

Average Precision:
0.922818

F1 Score:
0.879630

Precision:
0.818966

Recall:
0.950000

Train Loss:
0.049934

Validation Loss:
0.426662

------------------------------------------------------------
FINAL EPOCH RESULTS
------------------------------------------------------------

Final Epoch: 15

Validation Accuracy:
0.8550

Validation AUROC:
0.94110

Average Precision:
0.931144

F1 Score:
0.851282

Precision:
0.873684

Recall:
0.830000

------------------------------------------------------------
EXPERIMENT STATUS
------------------------------------------------------------

Status: COMPLETED
Source Type: experimental
Mock Data: False
Training Time: approximately 356.45 seconds

============================================================
"""

with open("/content/E1_RESULTS/E1_results_summary.txt", "w") as f:
    f.write(summary)

print("✅ E1 summary created")

✅ E1 summary created


In [95]:
import os

print("E1 RESULTS:")
print("=" * 60)

for file in os.listdir("/content/E1_RESULTS"):
    path = os.path.join("/content/E1_RESULTS", file)
    size = os.path.getsize(path)

    print(f"📄 {file}")
    print(f"   Size: {size:,} bytes")

E1 RESULTS:
📄 biggan_constrained_baseline_e1_best.pth
   Size: 282,599,413 bytes
📄 lota_experiments.db
   Size: 49,152 bytes
📄 biggan_constrained_baseline_e1.yaml
   Size: 696 bytes
📄 E1_results_summary.txt
   Size: 1,498 bytes


In [96]:
import shutil

shutil.make_archive(
    "/content/E1_RESULTS",
    "zip",
    "/content/E1_RESULTS"
)

print("✅ ZIP created")

✅ ZIP created


In [97]:
from google.colab import files

files.download("/content/E1_RESULTS.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>